# Lab: Secure Views and Stored Procedures

## Step1: Setup

In [ ]:
use role accountadmin;

create database if not exists weather;

use schema WEATHER.PUBLIC

In [ ]:
-- Create FILE FORMAT IF NOT EXISTS:
CREATE FILE FORMAT IF NOT EXISTS json_format
TYPE = 'JSON';

-- Create external stage object:
create stage IF NOT EXISTS nyc_weather
url = 's3://snowflake-workshop-lab/weather-nyc';

## Step 2: Create the Secure View

In [ ]:
CREATE OR REPLACE SECURE VIEW vw_weather AS
SELECT metadata$filename AS filename,
       metadata$file_row_number AS rnum,
       TO_VARIANT($1) AS v
FROM @nyc_weather (file_format => 'json_format');

## Step 3: Create the Secure Stored Procedure

In [ ]:
CREATE OR REPLACE SECURE PROCEDURE get_weather_data()
RETURNS TABLE (filename VARCHAR, rnum NUMBER, v VARIANT)
LANGUAGE SQL
EXECUTE AS OWNER
AS
$$
DECLARE
    res RESULTSET;
BEGIN
    res := (SELECT metadata$filename AS filename,
                   metadata$file_row_number AS rnum,
                   TO_VARIANT($1) AS v
            FROM @nyc_weather (file_format => 'json_format'));
    RETURN TABLE(res);
END;
$$;

In [ ]:
-- Check View DDL
SHOW VIEWS LIKE 'VW_WEATHER'->>SELECT "name", "is_secure", "text" FROM $1

In [ ]:
--Check DDL for the SP:

desc procedure GET_WEATHER_DATA() 
->> show procedures like 'GET_WEATHER_DATA'
->> SELECT 'Name' as key, "name" as value FROM $1 UNION ALL
    SELECT 'Is Secure', "is_secure" FROM $1 UNION ALL
    SELECT 'Body', "value" FROM $2 WHERE "property" = 'body'

## Step 4: Create a Test Role and Grant Privileges

In [ ]:
CREATE OR REPLACE ROLE test_role;

GRANT SELECT ON VIEW vw_weather TO ROLE test_role;
GRANT USAGE ON PROCEDURE get_weather_data() TO ROLE test_role;

GRANT USAGE ON DATABASE WEATHER TO ROLE test_role;
GRANT USAGE ON SCHEMA WEATHER.PUBLIC TO ROLE test_role;

GRANT ROLE test_role TO USER SCOTT202509; -- change the user name to yours (check CURRENT_USER())

## Step 5: Demonstrate DDL Protection

In [ ]:
USE ROLE test_role;
USE SECONDARY ROLES NONE; -- see the explanation below
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;

### What Are Secondary Roles?
In Snowflake, a user can have multiple roles granted to them (e.g., ACCOUNTADMIN, test_role, or others via GRANT ROLE). When you log in or switch roles using USE ROLE, the specified role becomes the primary role for your session. However:

Secondary roles are additional roles that are also active in the session, supplementing the primary role’s privileges.
By default, secondary roles can be enabled automatically based on your user’s granted roles or a session parameter (DEFAULT_SECONDARY_ROLES), unless explicitly managed.

### What Does USE SECONDARY ROLES NONE; Do?

Disables Secondary Roles: This command sets the session to use only the primary role (set by USE ROLE) and explicitly disables all secondary roles. No additional privileges from other roles are applied during the session.
Effect on Privileges: The session’s effective privileges are limited to those of the primary role, ignoring any privileges inherited from secondary roles. This is useful for isolating permissions and ensuring predictable behavior.
Session-Specific: The change applies only to the current session and does not affect the user’s default settings or other sessions.

In [ ]:
-- Check secure view DDL (should be hidden)
SELECT GET_DDL('VIEW', 'WEATHER.PUBLIC.VW_WEATHER');

In [ ]:
SHOW VIEWS LIKE 'VW_WEATHER'->>SELECT "name", "is_secure", "text" FROM $1

In [ ]:
-- Check secure procedure DDL (should be hidden)
SELECT GET_DDL('PROCEDURE', 'WEATHER.PUBLIC.GET_WEATHER_DATA()');

In [ ]:
EXECUTE IMMEDIATE $$
BEGIN
  SELECT GET_DDL('PROCEDURE', 'WEATHER.PUBLIC.GET_WEATHER_DATA()') as ddl;
EXCEPTION
  WHEN OTHER THEN
    LET LINE := SQLCODE || ': ' || SQLERRM;
    RETURN LINE;
END;
$$;

In [ ]:
desc procedure GET_WEATHER_DATA() 
->> show procedures like 'GET_WEATHER_DATA'
->> SELECT 'Name' as key, "name" as value FROM $1 UNION ALL
    SELECT 'Is Secure', "is_secure" FROM $1 UNION ALL
    SELECT 'Body', "value" FROM $2 WHERE "property" = 'body'


In [ ]:
desc procedure GET_WEATHER_DATA() 
->> show procedures like 'GET_WEATHER_DATA'
->> WITH b AS (
    SELECT "value" 
    FROM $2 
    WHERE "property"='body'
    )
    SELECT a."name", a."is_secure", b."value" as "body" 
    FROM $1 as a, b

## Step 6: Query the Objects

In [ ]:
USE ROLE test_role;
USE SECONDARY ROLES NONE;

-- Query the secure view
SELECT * FROM vw_weather LIMIT 2;

In [ ]:
-- Call the secure procedure
CALL get_weather_data() ->> SELECT * FROM $1 LIMIT 2;

In [ ]:
-- Use TABLE() syntax
SELECT * FROM TABLE(get_weather_data()) LIMIT 2;

### Key Takeaways

- DDL Protection: Secure views and procedures prevent non-owners from viewing the underlying query logic via GET_DDL or DESCRIBE, reducing risks like reverse-engineering or inference attacks.

- Best Practices: Always use SECURE for sensitive data. For procedures, EXECUTE AS OWNER encapsulates access without granting underlying privileges (e.g., to @nyc_weather).


## Step 7: Cleanup

In [ ]:
-- Cleanup (optional)

USE ROLE ACCOUNTADMIN;
DROP VIEW vw_weather;
DROP PROCEDURE get_weather_data();
DROP ROLE test_role;